In [1]:
import pandas as pd
# df.to_excel(r"D:\Dubai Data\Location leve summary\DataSales_with_Year_Quarter_final_BHK.xlsx", index=False)
df=pd.read_csv(r"D:\Dubai\Dubai_Updated_data\Transactions_Filtered.csv")
df

,transaction_id,procedure_id,trans_group_id,trans_group_ar,trans_group_en,procedure_name_ar,procedure_name_en,instance_date,property_type_id,property_type_ar,...,no_of_parties_role_1,no_of_parties_role_2,no_of_parties_role_3,Year,Quarter,Quarter_Year,property_type,procedure_area_sqft,sqft_sale_price,BHK
0,1-11-2013-15239,11,1,مبايعات,Sales,بيع,Sell,2013-05-16,4,فيلا,...,1.0,1.0,0.0,2013,Q2 2013,2013Q2,Others,4119.021613,485.552634,NaN
1,1-4-2012-13,4,1,مبايعات,Sales,إضافة أرض بالبيع,Adding Land By Sell,2012-04-19,2,مبنى,...,1.0,1.0,0.0,2012,Q2 2012,2012Q2,Others,3733.028159,246.636442,NaN
2,1-11-2006-316,11,1,مبايعات,Sales,بيع,Sell,2006-03-07,2,مبنى,...,9.0,1.0,0.0,2006,Q1 2006,2006Q1,Others,2274.950265,835.183344,NaN
3,1-11-2026-121,11,1,مبايعات,Sales,بيع,Sell,2026-01-05,2,مبنى,...,13.0,1.0,0.0,2026,Q1 2026,2026Q1,Others,2503.252584,3599.716645,NaN
4,1-11-2025-37635,11,1,مبايعات,Sales,بيع,Sell,2025-09-09,2,مبنى,...,1.0,1.0,0.0,2025,Q3 2025,2025Q3,Others,2276.672489,5879.633776,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1167307,1-11-2013-19796,11,1,مبايعات,Sales,بيع,Sell,2013-06-26,3,وحدة,...,1.0,1.0,0.0,2013,Q2 2013,2013Q2,Flat,2279.901659,1820.253811,3 B/R
1167308,1-45-2017-404,45,1,مبايعات,Sales,بيع حق منفعة,Sell Development,2017-06-12,2,مبنى,...,1.0,1.0,0.0,2017,Q2 2017,2017Q2,Others,58456.157564,297.658841,NaN
1167309,1-11-2024-10269,11,1,مبايعات,Sales,بيع,Sell,2024-03-20,3,وحدة,...,1.0,1.0,0.0,2024,Q1 2024,2024Q1,Flat,1338.168048,2391.329351,1 B/R
1167310,1-11-2012-23048,11,1,مبايعات,Sales,بيع,Sell,2012-12-05,4,فيلا,...,1.0,1.0,0.0,2012,Q4 2012,2012Q4,Others,5692.596154,869.550070,NaN


In [2]:
df.columns

Index(['transaction_id', 'procedure_id', 'trans_group_id', 'trans_group_ar',
       'trans_group_en', 'procedure_name_ar', 'procedure_name_en',
       'instance_date', 'property_type_id', 'property_type_ar',
       'property_type_en', 'property_sub_type_id', 'property_sub_type_ar',
       'property_sub_type_en', 'property_usage_ar', 'property_usage_en',
       'reg_type_id', 'reg_type_ar', 'reg_type_en', 'area_id', 'area_name_ar',
       'area_name_en', 'building_name_ar', 'building_name_en',
       'project_number', 'project_name_ar', 'project_name_en',
       'master_project_en', 'master_project_ar', 'nearest_landmark_ar',
       'nearest_landmark_en', 'nearest_metro_ar', 'nearest_metro_en',
       'nearest_mall_ar', 'nearest_mall_en', 'rooms_ar', 'rooms_en',
       'has_parking', 'procedure_area', 'actual_worth', 'meter_sale_price',
       'rent_value', 'meter_rent_price', 'no_of_parties_role_1',
       'no_of_parties_role_2', 'no_of_parties_role_3', 'Year', 'Quarter',
       'Quart

In [3]:
import numpy as np
def create_integrated_comprehensive_analysis(df, output_file='complete_integrated_analysis.xlsx'):
    """
    Create COMPLETE integrated analysis with ALL columns including FULL BHK breakdowns:
    
    1. Base summary with ALL metrics
    2. Property Type breakdowns (Units, Area, Sales, Rate stats)
    3. BHK breakdowns (Units, Area, Sales, Rate stats) - ALL metrics
    4. Range analysis (Rate, Area, Price ranges)
    5. Property Type range breakdowns
    6. BHK range breakdowns
    7. In 6 sheets: Area (3) + City (3)
    """
    
    print("="*80)
    print("CREATING INTEGRATED COMPREHENSIVE ANALYSIS WITH FULL BHK BREAKDOWNS")
    print("="*80)
    
    # Prepare data
    df_work = df.copy()
    
    # Ensure Date exists
    if 'instance_date' in df_work.columns and 'Date' not in df_work.columns:
        df_work['Date'] = pd.to_datetime(df_work['instance_date'], errors='coerce')
    
    # Get city column name
    city_col = None
    city_candidates = ['city_name_en', 'city', 'City', 'CITY_NAME', 'location_city']
    for col in city_candidates:
        if col in df_work.columns:
            city_col = col
            print(f"Using city column: {city_col}")
            break
    
    if not city_col:
        print("WARNING: No city column found. Creating dummy city column.")
        df_work['city'] = 'DUBAI'  # Default
        city_col = 'city'
    
    df_work = df_work.dropna(subset=['Date', 'area_name_en', city_col])
    print(f"Rows after cleaning = {len(df_work)}")
    
    # Get key columns
    area_col = 'procedure_area_sqft' if 'procedure_area_sqft' in df_work.columns else None
    price_sqft_col = 'sqft_sale_price' if 'sqft_sale_price' in df_work.columns else None
    total_price_col = 'actual_worth' if 'actual_worth' in df_work.columns else None
    property_type_col = 'property_type'
    bhk_col = 'BHK'
    
    # Calculate total sales
    if area_col and price_sqft_col:
        df_work['Total_Sales_Sqft'] = df_work[area_col] * df_work[price_sqft_col]
    elif total_price_col:
        df_work['Total_Sales_Sqft'] = df_work[total_price_col]
    
    # Create time columns
    df_work['Year'] = df_work['Date'].dt.year.astype(int)
    df_work['Quarter_Num'] = df_work['Date'].dt.quarter.astype(int)
    df_work['Quarter'] = 'Q' + df_work['Quarter_Num'].astype(str) + ' ' + df_work['Year'].astype(str)
    df_work['city_name_en'] = df_work[city_col]
    
    # ============================================
    # 1. CREATE RANGE BINS
    # ============================================
    
    # Rate bins (sqft price): <3000 to >40000, 1000 interval
    def categorize_rate(rate):
        if pd.isna(rate) or price_sqft_col is None:
            return 'Unknown'
        rate_val = float(rate)
        if rate_val < 350: return '<350'
        if rate_val > 6300: return '>6300'
        bins = list(range(350, 6300, 350))
        for i in range(len(bins)-1):
            if bins[i] <= rate_val < bins[i+1]:
                return f'{bins[i]}-{bins[i+1]}'
        return 'Unknown'
    
    if price_sqft_col:
        df_work['Rate_Range'] = df_work[price_sqft_col].apply(categorize_rate)
        rate_bins = ['<350'] + [f'{i}-{i+350}' for i in range(350, 6300, 350)] + ['>6300']
    
    # Area bins: <200 to >6500, 300 interval
    def categorize_area(area):
        if pd.isna(area) or area_col is None:
            return 'Unknown'
        area_val = float(area)
        if area_val < 300: return '<300'
        if area_val > 3000: return '>3000'
        bins = list(range(300, 6800, 300))
        for i in range(len(bins)-1):
            if bins[i] <= area_val < bins[i+1]:
                return f'{bins[i]}-{bins[i+1]}'
        return 'Unknown'
    
    if area_col:
        df_work['Area_Range'] = df_work[area_col].apply(categorize_area)
        area_bins = ['<300'] + [f'{i}-{i+300}' for i in range(300, 3000, 300)] + ['>3000']
    
    # Price bins (total worth): <5M to >150M, 5M interval
    # Price bins (total worth): from <350000 to >52500000, with 350000 intervals
    def categorize_price(price):
        if pd.isna(price) or total_price_col is None:
            return 'Unknown'
        price_val = float(price)
        
        # Handle the edge cases
        if price_val < 350000:
            return '<350000'
        if price_val > 52500000:
            return '>52500000'
        
        # Create bins from 350000 to 52500000 with 350000 intervals
        for bin_start in range(350000, 52500000, 350000):
            bin_end = bin_start + 350000
            if bin_start <= price_val < bin_end:
                return f'{bin_start:,}-{bin_end:,}'
        
        return 'Unknown'

    if total_price_col:
        df_work['Price_Range'] = df_work[total_price_col].apply(categorize_price)
        # Create the bin labels for reference
        price_bins = ['<350000'] + [f'{i:,}-{i+350000:,}' for i in range(350000, 52500000, 350000)] + ['>52500000']

        
    # ============================================
    # 2. HELPER FUNCTIONS FOR RANGE ANALYSIS
    # ============================================
    
    def create_range_dicts(data, range_col, range_bins, prefix):
        """Create dictionaries for range analysis - TOTAL values"""
        if range_col not in data.columns:
            return {}
        
        units_dict = {bin_label: 0 for bin_label in range_bins}
        area_dict = {bin_label: 0 for bin_label in range_bins}
        sales_dict = {bin_label: 0 for bin_label in range_bins}
        
        for range_val, group in data.groupby(range_col):
            if range_val in units_dict:
                units_dict[range_val] = int(len(group))
                if area_col:
                    area_dict[range_val] = float(group[area_col].sum())
                if 'Total_Sales_Sqft' in group.columns:
                    sales_dict[range_val] = float(group['Total_Sales_Sqft'].sum())
        
        def dict_to_string(d):
            filtered = {k: v for k, v in d.items() if v != 0}
            if not filtered: return '{}'
            items = [f"'{k}':{v}" for k, v in filtered.items()]
            return '{' + ','.join(items) + '}'
        
        return {
            f'{prefix}_Units_Sold': dict_to_string(units_dict),
            f'{prefix}_Area_Consumed': dict_to_string(area_dict) if area_col else '{}',
            f'{prefix}_Total_Sales': dict_to_string(sales_dict)
        }
    
    def create_category_range_dicts_enhanced(data, range_col, range_bins, category_col, prefix):
        """Create COMPLETE nested dictionaries for category breakdowns with ALL metrics"""
        if range_col not in data.columns or category_col not in data.columns:
            return {}
        
        units_result = {}
        area_result = {}
        sales_result = {}
        
        categories = data[category_col].unique()
        
        for cat in categories:
            cat_data = data[data[category_col] == cat]
            
            units_dict = {bin_label: 0 for bin_label in range_bins}
            area_dict = {bin_label: 0 for bin_label in range_bins}
            sales_dict = {bin_label: 0 for bin_label in range_bins}
            
            for range_val, group in cat_data.groupby(range_col):
                if range_val in units_dict:
                    units_dict[range_val] = int(len(group))
                    if area_col:
                        area_dict[range_val] = float(group[area_col].sum())
                    if 'Total_Sales_Sqft' in group.columns:
                        sales_dict[range_val] = float(group['Total_Sales_Sqft'].sum())
            
            if any(v > 0 for v in units_dict.values()):
                def filter_dict(d):
                    return {k: v for k, v in d.items() if v > 0}
                
                filtered_units = filter_dict(units_dict)
                filtered_area = filter_dict(area_dict)
                filtered_sales = filter_dict(sales_dict)
                
                if filtered_units:
                    units_result[str(cat)] = filtered_units
                if filtered_area:
                    area_result[str(cat)] = filtered_area
                if filtered_sales:
                    sales_result[str(cat)] = filtered_sales
        
        def dict_to_nested_string(nested_dict):
            if not nested_dict: return '{}'
            items = []
            for cat, range_dict in nested_dict.items():
                range_items = [f"'{k}':{v}" for k, v in range_dict.items()]
                items.append(f"'{cat}':{{{','.join(range_items)}}}")
            return '{' + ','.join(items) + '}'
        
        result = {}
        
        range_type = range_col.split('_')[0]  # 'Rate', 'Area', or 'Price'
        
        if units_result:
            result[f'{prefix}_{range_type}_Units'] = dict_to_nested_string(units_result)
        if area_result and area_col:
            result[f'{prefix}_{range_type}_Area'] = dict_to_nested_string(area_result)
        if sales_result:
            result[f'{prefix}_{range_type}_Sales'] = dict_to_nested_string(sales_result)
        
        return result
    
    # ============================================
    # 3. CREATE BASE SUMMARY WITH FULL BREAKDOWNS
    # ============================================
    
    def create_group_summary_complete(df_group, group_col, time_col=None):
        """Create COMPLETE base summary with ALL breakdowns for any grouping"""
        if time_col:
            index_cols = [group_col, time_col]
        else:
            index_cols = [group_col]
        
        # Clean category names for column naming
        def clean_name(name):
            if pd.isna(name):
                return 'Unknown'
            name = str(name)
            name = name.replace(' ', '_').replace('/', '_').replace('\\', '_')
            name = name.replace('-', '_').replace('&', 'and').replace('(', '').replace(')', '')
            name = name.replace('+', 'plus').replace('%', 'pct').replace('#', 'num')
            name = name.replace('.', '_').replace(',', '_').replace(';', '_')
            name = name.replace('__', '_').replace('__', '_')
            return name.strip('_').upper()
        
        # Create cleaned category columns
        df_clean = df_group.copy()
        if property_type_col in df_clean.columns:
            df_clean['property_type_clean'] = df_clean[property_type_col].apply(clean_name)
        if bhk_col in df_clean.columns:
            df_clean['bhk_clean'] = df_clean[bhk_col].apply(lambda x: clean_name(x))
        
        # Base metrics
        base_agg = {
            'Total_Units_Sold': ('transaction_id', 'count'),
        }
        
        if area_col:
            base_agg['Total_Area_Consumed'] = (area_col, 'sum')
        
        if price_sqft_col:
            base_agg.update({
                'Avg_Rate_per_Sqft': (price_sqft_col, 'mean'),
                'Rate_50th_Percentile': (price_sqft_col, lambda x: x.quantile(0.50)),
                'Rate_75th_Percentile': (price_sqft_col, lambda x: x.quantile(0.75)),
                'Rate_90th_Percentile': (price_sqft_col, lambda x: x.quantile(0.90))
            })
        
        if area_col and price_sqft_col:
            base_agg['Total_Sales_Value'] = ('Total_Sales_Sqft', 'sum')
        
        sheet_data = df_clean.groupby(index_cols).agg(**base_agg).reset_index()
        
        # ============================================
        # ADD PROPERTY TYPE BREAKDOWNS (COMPLETE)
        # ============================================
        
        if 'property_type_clean' in df_clean.columns:
            property_types = df_clean['property_type_clean'].unique()
            
            for prop_type in property_types:
                prop_data = df_clean[df_clean['property_type_clean'] == prop_type]
                
                # Units by property type
                units_pivot = prop_data.groupby(index_cols).size().reset_index(name=f'Prop_{prop_type}_Units_Sold')
                sheet_data = sheet_data.merge(units_pivot, on=index_cols, how='left')
                
                # Area by property type
                if area_col:
                    area_pivot = prop_data.groupby(index_cols)[area_col].sum().reset_index(name=f'Prop_{prop_type}_Area_Consumed')
                    sheet_data = sheet_data.merge(area_pivot, on=index_cols, how='left')
                
                # Sales by property type
                if area_col and price_sqft_col:
                    sales_pivot = prop_data.groupby(index_cols)['Total_Sales_Sqft'].sum().reset_index(name=f'Prop_{prop_type}_Total_Sales')
                    sheet_data = sheet_data.merge(sales_pivot, on=index_cols, how='left')
                
                # Rate metrics by property type
                if price_sqft_col:
                    rate_metrics = ['mean', lambda x: x.quantile(0.50), lambda x: x.quantile(0.75), lambda x: x.quantile(0.90)]
                    suffixes = ['Avg_Rate', 'Rate_50th', 'Rate_75th', 'Rate_90th']
                    
                    for metric, suffix in zip(rate_metrics, suffixes):
                        rate_pivot = prop_data.groupby(index_cols)[price_sqft_col].apply(metric).reset_index(name=f'Prop_{prop_type}_{suffix}')
                        sheet_data = sheet_data.merge(rate_pivot, on=index_cols, how='left')
        
        # ============================================
        # ADD BHK BREAKDOWNS (COMPLETE - ALL METRICS)
        # ============================================
        
        if 'bhk_clean' in df_clean.columns:
            bhk_types = df_clean['bhk_clean'].unique()
            
            for bhk_type in bhk_types:
                bhk_data = df_clean[df_clean['bhk_clean'] == bhk_type]
                
                # Units by BHK
                units_pivot = bhk_data.groupby(index_cols).size().reset_index(name=f'BHK_{bhk_type}_Units_Sold')
                sheet_data = sheet_data.merge(units_pivot, on=index_cols, how='left')
                
                # Area by BHK
                if area_col:
                    area_pivot = bhk_data.groupby(index_cols)[area_col].sum().reset_index(name=f'BHK_{bhk_type}_Area_Consumed')
                    sheet_data = sheet_data.merge(area_pivot, on=index_cols, how='left')
                
                # Sales by BHK
                if area_col and price_sqft_col:
                    sales_pivot = bhk_data.groupby(index_cols)['Total_Sales_Sqft'].sum().reset_index(name=f'BHK_{bhk_type}_Total_Sales')
                    sheet_data = sheet_data.merge(sales_pivot, on=index_cols, how='left')
                
                # Rate metrics by BHK (ONLY if there are enough data points)
                if price_sqft_col:
                    # Check if we have enough data for rate calculations
                    if len(bhk_data) >= 3:  # Minimum 3 records for meaningful stats
                        rate_metrics = ['mean', lambda x: x.quantile(0.50), lambda x: x.quantile(0.75), lambda x: x.quantile(0.90)]
                        suffixes = ['Avg_Rate', 'Rate_50th', 'Rate_75th', 'Rate_90th']
                        
                        for metric, suffix in zip(rate_metrics, suffixes):
                            try:
                                rate_pivot = bhk_data.groupby(index_cols)[price_sqft_col].apply(metric).reset_index(name=f'BHK_{bhk_type}_{suffix}')
                                sheet_data = sheet_data.merge(rate_pivot, on=index_cols, how='left')
                            except:
                                # If calculation fails, add empty column
                                sheet_data[f'BHK_{bhk_type}_{suffix}'] = 0
                    else:
                        # Add empty rate columns for small sample sizes
                        suffixes = ['Avg_Rate', 'Rate_50th', 'Rate_75th', 'Rate_90th']
                        for suffix in suffixes:
                            sheet_data[f'BHK_{bhk_type}_{suffix}'] = 0
        
        # Fill NaN values
        sheet_data = sheet_data.fillna(0)
        
        # Add Market Share Percentage
        if 'Total_Units_Sold' in sheet_data.columns:
            if time_col:
                sheet_data['Market_Share_Pct'] = (
                    sheet_data.groupby(time_col)['Total_Units_Sold']
                    .transform(lambda x: (x / x.sum() * 100).round(2) if x.sum() > 0 else 0)
                )
            else:
                total_units = sheet_data['Total_Units_Sold'].sum()
                sheet_data['Market_Share_Pct'] = (
                    sheet_data['Total_Units_Sold'] / total_units * 100
                ).round(2) if total_units > 0 else 0
        
        return sheet_data
    
    # ============================================
    # 4. CREATE ALL 6 SHEETS
    # ============================================
    
    print("\n" + "="*50)
    print("CREATING ALL 6 SHEETS WITH FULL BREAKDOWNS")
    print("="*50)
    
    all_sheets = {}
    
    # AREA SHEETS (3)
    for sheet_type, time_col in [('Overall', None), ('YoY', 'Year'), ('QoQ', 'Quarter')]:
        print(f"\nCreating Area {sheet_type} sheet...")
        
        # Get base summary WITH ALL BREAKDOWNS
        sheet_data = create_group_summary_complete(df_work, 'area_name_en', time_col)
        
        # Sort data
        if time_col:
            sheet_data = sheet_data.sort_values([time_col, 'Total_Units_Sold'], ascending=[True, False])
        else:
            sheet_data = sheet_data.sort_values('Total_Units_Sold', ascending=False)
        
        # Add range analysis columns
        range_columns = []
        
        if price_sqft_col:
            range_columns.extend([
                'Rate_Range_Units_Sold', 'Rate_Range_Area_Consumed', 'Rate_Range_Total_Sales',
                'PropType_Rate_Units', 'PropType_Rate_Area', 'PropType_Rate_Sales',
                'BHK_Rate_Units', 'BHK_Rate_Area', 'BHK_Rate_Sales'
            ])
        
        if area_col:
            range_columns.extend([
                'Area_Range_Units_Sold', 'Area_Range_Area_Consumed', 'Area_Range_Total_Sales',
                'PropType_Area_Units', 'PropType_Area_Area', 'PropType_Area_Sales',
                'BHK_Area_Units', 'BHK_Area_Area', 'BHK_Area_Sales'
            ])
        
        if total_price_col:
            range_columns.extend([
                'Price_Range_Units_Sold', 'Price_Range_Area_Consumed', 'Price_Range_Total_Sales',
                'PropType_Price_Units', 'PropType_Price_Area', 'PropType_Price_Sales',
                'BHK_Price_Units', 'BHK_Price_Area', 'BHK_Price_Sales'
            ])
        
        # Add empty range columns
        for col in range_columns:
            if col not in sheet_data.columns:
                sheet_data[col] = ''
        
        # Add range analysis data for each row
        for idx, row in sheet_data.iterrows():
            area_name = row['area_name_en']
            
            # Skip totals row for now
            if area_name in ['ALL AREAS TOTAL', 'ALL AREAS']:
                continue
            
            # Filter data
            if sheet_type == 'Overall':
                group_data = df_work[df_work['area_name_en'] == area_name]
            elif sheet_type == 'YoY' and 'Year' in row:
                year = row['Year']
                group_data = df_work[(df_work['area_name_en'] == area_name) & (df_work['Year'] == year)]
            elif sheet_type == 'QoQ' and 'Quarter' in row:
                quarter = row['Quarter']
                group_data = df_work[(df_work['area_name_en'] == area_name) & (df_work['Quarter'] == quarter)]
            else:
                continue
            
            if len(group_data) == 0:
                continue
            
            # Add all range analyses
            all_dicts = {}
            
            if price_sqft_col:
                all_dicts.update(create_range_dicts(group_data, 'Rate_Range', rate_bins, 'RateRange'))
                if property_type_col in group_data.columns:
                    all_dicts.update(create_category_range_dicts_enhanced(group_data, 'Rate_Range', rate_bins, property_type_col, 'PropType'))
                if bhk_col in group_data.columns:
                    all_dicts.update(create_category_range_dicts_enhanced(group_data, 'Rate_Range', rate_bins, bhk_col, 'BHK'))
            
            if area_col:
                all_dicts.update(create_range_dicts(group_data, 'Area_Range', area_bins, 'AreaRange'))
                if property_type_col in group_data.columns:
                    all_dicts.update(create_category_range_dicts_enhanced(group_data, 'Area_Range', area_bins, property_type_col, 'PropType'))
                if bhk_col in group_data.columns:
                    all_dicts.update(create_category_range_dicts_enhanced(group_data, 'Area_Range', area_bins, bhk_col, 'BHK'))
            
            if total_price_col:
                all_dicts.update(create_range_dicts(group_data, 'Price_Range', price_bins, 'PriceRange'))
                if property_type_col in group_data.columns:
                    all_dicts.update(create_category_range_dicts_enhanced(group_data, 'Price_Range', price_bins, property_type_col, 'PropType'))
                if bhk_col in group_data.columns:
                    all_dicts.update(create_category_range_dicts_enhanced(group_data, 'Price_Range', price_bins, bhk_col, 'BHK'))
            
            # Update row
            for col, val in all_dicts.items():
                sheet_data.at[idx, col] = val
        
        # Add totals row
        totals_row = {}
        totals_row['area_name_en'] = 'ALL AREAS TOTAL'
        if time_col:
            if sheet_type == 'YoY':
                totals_row['Year'] = 'ALL YEARS'
            elif sheet_type == 'QoQ':
                totals_row['Quarter'] = 'ALL QUARTERS'
        
        # Calculate totals for numeric columns
        numeric_cols = sheet_data.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if col != 'Market_Share_Pct' and not col.startswith(('Prop_', 'BHK_')):
                # Sum base numeric columns
                totals_row[col] = sheet_data[col].sum()
            elif col.startswith(('Prop_', 'BHK_')):
                # For breakdown columns, sum them too
                totals_row[col] = sheet_data[col].sum()
            elif col == 'Market_Share_Pct':
                totals_row[col] = 100.0
        
        # Add range totals
        totals_data = df_work
        totals_dicts = {}
        
        if price_sqft_col:
            totals_dicts.update(create_range_dicts(totals_data, 'Rate_Range', rate_bins, 'RateRange'))
            if property_type_col in totals_data.columns:
                totals_dicts.update(create_category_range_dicts_enhanced(totals_data, 'Rate_Range', rate_bins, property_type_col, 'PropType'))
            if bhk_col in totals_data.columns:
                totals_dicts.update(create_category_range_dicts_enhanced(totals_data, 'Rate_Range', rate_bins, bhk_col, 'BHK'))
        
        if area_col:
            totals_dicts.update(create_range_dicts(totals_data, 'Area_Range', area_bins, 'AreaRange'))
            if property_type_col in totals_data.columns:
                totals_dicts.update(create_category_range_dicts_enhanced(totals_data, 'Area_Range', area_bins, property_type_col, 'PropType'))
            if bhk_col in totals_data.columns:
                totals_dicts.update(create_category_range_dicts_enhanced(totals_data, 'Area_Range', area_bins, bhk_col, 'BHK'))
        
        if total_price_col:
            totals_dicts.update(create_range_dicts(totals_data, 'Price_Range', price_bins, 'PriceRange'))
            if property_type_col in totals_data.columns:
                totals_dicts.update(create_category_range_dicts_enhanced(totals_data, 'Price_Range', price_bins, property_type_col, 'PropType'))
            if bhk_col in totals_data.columns:
                totals_dicts.update(create_category_range_dicts_enhanced(totals_data, 'Price_Range', price_bins, bhk_col, 'BHK'))
        
        totals_row.update(totals_dicts)
        
        # Append totals row
        sheet_data = pd.concat([sheet_data, pd.DataFrame([totals_row])], ignore_index=True)
        
        all_sheets[sheet_type] = sheet_data
    
    # CITY SHEETS (3)
    for sheet_type, time_col in [('City_Overall', None), ('City_YoY', 'Year'), ('City_QoQ', 'Quarter')]:
        print(f"\nCreating {sheet_type} sheet...")
        
        # Get base summary WITH ALL BREAKDOWNS
        sheet_data = create_group_summary_complete(df_work, 'city_name_en', time_col)
        
        # Sort data
        if time_col:
            sheet_data = sheet_data.sort_values([time_col, 'Total_Units_Sold'], ascending=[True, False])
        else:
            sheet_data = sheet_data.sort_values('Total_Units_Sold', ascending=False)
        
        # Add range analysis columns (same as area sheets)
        range_columns = []
        
        if price_sqft_col:
            range_columns.extend([
                'RateRange_Units_Sold', 'RateRange_Area_Consumed', 'RateRange_Total_Sales',
                'PropType_Rate_Units', 'PropType_Rate_Area', 'PropType_Rate_Sales',
                'BHK_Rate_Units', 'BHK_Rate_Area', 'BHK_Rate_Sales'
            ])
        
        if area_col:
            range_columns.extend([
                'AreaRange_Units_Sold', 'AreaRange_Area_Consumed', 'AreaRange_Total_Sales',
                'PropType_Area_Units', 'PropType_Area_Area', 'PropType_Area_Sales',
                'BHK_Area_Units', 'BHK_Area_Area', 'BHK_Area_Sales'
            ])
        
        if total_price_col:
            range_columns.extend([
                'PriceRange_Units_Sold', 'PriceRange_Area_Consumed', 'PriceRange_Total_Sales',
                'PropType_Price_Units', 'PropType_Price_Area', 'PropType_Price_Sales',
                'BHK_Price_Units', 'BHK_Price_Area', 'BHK_Price_Sales'
            ])
        
        # Add empty range columns
        for col in range_columns:
            if col not in sheet_data.columns:
                sheet_data[col] = ''
        
        # Add range analysis data for each row
        for idx, row in sheet_data.iterrows():
            city_name = row['city_name_en']
            
            # Skip totals row for now
            if city_name in ['ALL CITIES TOTAL', 'ALL CITIES']:
                continue
            
            # Filter data
            if sheet_type == 'City_Overall':
                group_data = df_work[df_work['city_name_en'] == city_name]
            elif sheet_type == 'City_YoY' and 'Year' in row:
                year = row['Year']
                group_data = df_work[(df_work['city_name_en'] == city_name) & (df_work['Year'] == year)]
            elif sheet_type == 'City_QoQ' and 'Quarter' in row:
                quarter = row['Quarter']
                group_data = df_work[(df_work['city_name_en'] == city_name) & (df_work['Quarter'] == quarter)]
            else:
                continue
            
            if len(group_data) == 0:
                continue
            
            # Add all range analyses
            all_dicts = {}
            
            if price_sqft_col:
                all_dicts.update(create_range_dicts(group_data, 'Rate_Range', rate_bins, 'RateRange'))
                if property_type_col in group_data.columns:
                    all_dicts.update(create_category_range_dicts_enhanced(group_data, 'Rate_Range', rate_bins, property_type_col, 'PropType'))
                if bhk_col in group_data.columns:
                    all_dicts.update(create_category_range_dicts_enhanced(group_data, 'Rate_Range', rate_bins, bhk_col, 'BHK'))
            
            if area_col:
                all_dicts.update(create_range_dicts(group_data, 'Area_Range', area_bins, 'AreaRange'))
                if property_type_col in group_data.columns:
                    all_dicts.update(create_category_range_dicts_enhanced(group_data, 'Area_Range', area_bins, property_type_col, 'PropType'))
                if bhk_col in group_data.columns:
                    all_dicts.update(create_category_range_dicts_enhanced(group_data, 'Area_Range', area_bins, bhk_col, 'BHK'))
            
            if total_price_col:
                all_dicts.update(create_range_dicts(group_data, 'Price_Range', price_bins, 'PriceRange'))
                if property_type_col in group_data.columns:
                    all_dicts.update(create_category_range_dicts_enhanced(group_data, 'Price_Range', price_bins, property_type_col, 'PropType'))
                if bhk_col in group_data.columns:
                    all_dicts.update(create_category_range_dicts_enhanced(group_data, 'Price_Range', price_bins, bhk_col, 'BHK'))
            
            # Update row
            for col, val in all_dicts.items():
                sheet_data.at[idx, col] = val
        
        # Add totals row
        totals_row = {}
        totals_row['city_name_en'] = 'ALL CITIES TOTAL'
        if time_col:
            if sheet_type == 'City_YoY':
                totals_row['Year'] = 'ALL YEARS'
            elif sheet_type == 'City_QoQ':
                totals_row['Quarter'] = 'ALL QUARTERS'
        
        # Calculate totals for numeric columns
        numeric_cols = sheet_data.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if col != 'Market_Share_Pct' and not col.startswith(('Prop_', 'BHK_')):
                # Sum base numeric columns
                totals_row[col] = sheet_data[col].sum()
            elif col.startswith(('Prop_', 'BHK_')):
                # For breakdown columns, sum them too
                totals_row[col] = sheet_data[col].sum()
            elif col == 'Market_Share_Pct':
                totals_row[col] = 100.0
        
        # Add range totals
        totals_data = df_work
        totals_dicts = {}
        
        if price_sqft_col:
            totals_dicts.update(create_range_dicts(totals_data, 'Rate_Range', rate_bins, 'RateRange'))
            if property_type_col in totals_data.columns:
                totals_dicts.update(create_category_range_dicts_enhanced(totals_data, 'Rate_Range', rate_bins, property_type_col, 'PropType'))
            if bhk_col in totals_data.columns:
                totals_dicts.update(create_category_range_dicts_enhanced(totals_data, 'Rate_Range', rate_bins, bhk_col, 'BHK'))
        
        if area_col:
            totals_dicts.update(create_range_dicts(totals_data, 'Area_Range', area_bins, 'AreaRange'))
            if property_type_col in totals_data.columns:
                totals_dicts.update(create_category_range_dicts_enhanced(totals_data, 'Area_Range', area_bins, property_type_col, 'PropType'))
            if bhk_col in totals_data.columns:
                totals_dicts.update(create_category_range_dicts_enhanced(totals_data, 'Area_Range', area_bins, bhk_col, 'BHK'))
        
        if total_price_col:
            totals_dicts.update(create_range_dicts(totals_data, 'Price_Range', price_bins, 'PriceRange'))
            if property_type_col in totals_data.columns:
                totals_dicts.update(create_category_range_dicts_enhanced(totals_data, 'Price_Range', price_bins, property_type_col, 'PropType'))
            if bhk_col in totals_data.columns:
                totals_dicts.update(create_category_range_dicts_enhanced(totals_data, 'Price_Range', price_bins, bhk_col, 'BHK'))
        
        totals_row.update(totals_dicts)
        
        # Append totals row
        sheet_data = pd.concat([sheet_data, pd.DataFrame([totals_row])], ignore_index=True)
        
        all_sheets[sheet_type] = sheet_data
    
    # ============================================
    # 5. SAVE TO EXCEL
    # ============================================
    
    print(f"\nSaving to {output_file}...")
    
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        # Save Area sheets
        for sheet_name in ['Overall', 'YoY', 'QoQ']:
            if sheet_name in all_sheets:
                all_sheets[sheet_name].to_excel(writer, sheet_name=sheet_name, index=False)
        
        # Save City sheets
        for sheet_name in ['City_Overall', 'City_YoY', 'City_QoQ']:
            if sheet_name in all_sheets:
                all_sheets[sheet_name].to_excel(writer, sheet_name=sheet_name, index=False)
        
        # Adjust column widths
        for sheet_name in writer.sheets:
            ws = writer.sheets[sheet_name]
            for column in ws.columns:
                max_length = 0
                column_letter = column[0].column_letter
                for cell in column:
                    try:
                        if cell.value and len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                ws.column_dimensions[column_letter].width = min(max_length + 2, 30)
    
    print("\n" + "="*80)
    print("INTEGRATED COMPREHENSIVE ANALYSIS WITH FULL BHK BREAKDOWNS COMPLETE!")
    print("="*80)
    
    # Show sample of columns created
    if 'Overall' in all_sheets:
        sample_df = all_sheets['Overall']
        print(f"\nCOLUMNS CREATED ({len(sample_df.columns)} total):")
        print("-" * 60)
        
        # Group columns by type
        base_cols = [col for col in sample_df.columns if not col.startswith(('Prop_', 'BHK_', 'RateRange', 'AreaRange', 'PriceRange'))]
        prop_cols = [col for col in sample_df.columns if col.startswith('Prop_')]
        bhk_cols = [col for col in sample_df.columns if col.startswith('BHK_') and not 'BHK_Rate' in col and not 'BHK_Area' in col and not 'BHK_Price' in col]
        range_cols = [col for col in sample_df.columns if col.startswith(('RateRange', 'AreaRange', 'PriceRange'))]
        prop_range_cols = [col for col in sample_df.columns if col.startswith('PropType_')]
        bhk_range_cols = [col for col in sample_df.columns if col.startswith('BHK_') and ('Rate' in col or 'Area' in col or 'Price' in col)]
        
        print(f"1. Base Metrics: {len(base_cols)} columns")
        print(f"   Samples: {base_cols[:5]}...")
        
        print(f"\n2. Property Type Breakdowns: {len(prop_cols)} columns")
        prop_units = [col for col in prop_cols if 'Units_Sold' in col]
        prop_area = [col for col in prop_cols if 'Area_Consumed' in col]
        prop_sales = [col for col in prop_cols if 'Total_Sales' in col]
        prop_rates = [col for col in prop_cols if 'Rate' in col]
        print(f"   - Units: {len(prop_units)} columns")
        print(f"   - Area: {len(prop_area)} columns")
        print(f"   - Sales: {len(prop_sales)} columns")
        print(f"   - Rate stats: {len(prop_rates)} columns")
        
        print(f"\n3. BHK Breakdowns: {len(bhk_cols)} columns")
        bhk_units = [col for col in bhk_cols if 'Units_Sold' in col]
        bhk_area = [col for col in bhk_cols if 'Area_Consumed' in col]
        bhk_sales = [col for col in bhk_cols if 'Total_Sales' in col]
        bhk_rates = [col for col in bhk_cols if 'Rate' in col]
        print(f"   - Units: {len(bhk_units)} columns (BHK_1_BHK_Units_Sold, etc.)")
        print(f"   - Area: {len(bhk_area)} columns (BHK_1_BHK_Area_Consumed, etc.)")
        print(f"   - Sales: {len(bhk_sales)} columns (BHK_1_BHK_Total_Sales, etc.)")
        print(f"   - Rate stats: {len(bhk_rates)} columns (BHK_1_BHK_Avg_Rate, etc.)")
        
        print(f"\n4. Range Analysis: {len(range_cols) + len(prop_range_cols) + len(bhk_range_cols)} columns")
        print(f"   - Total Ranges: {len(range_cols)}")
        print(f"   - Property Type Ranges: {len(prop_range_cols)}")
        print(f"   - BHK Ranges: {len(bhk_range_cols)}")
    
    print(f"\nOutput file: {output_file}")
    print("="*80)
    
    return output_file


# ============================================
# MAIN EXECUTION
# ============================================

if __name__ == "__main__":
    print("Integrated comprehensive analysis function ready!")
    print("\nUsage:")
    print("final_output = create_integrated_comprehensive_analysis(df, 'final_complete_analysis.xlsx')")
    final_output = create_integrated_comprehensive_analysis(df, 'final_complete_analysis.xlsx')

Integrated comprehensive analysis function ready!

Usage:
final_output = create_integrated_comprehensive_analysis(df, 'final_complete_analysis.xlsx')
CREATING INTEGRATED COMPREHENSIVE ANALYSIS WITH FULL BHK BREAKDOWNS
Rows after cleaning = 1167312

CREATING ALL 6 SHEETS WITH FULL BREAKDOWNS

Creating Area Overall sheet...

Creating Area YoY sheet...

Creating Area QoQ sheet...

Creating City_Overall sheet...

Creating City_YoY sheet...

Creating City_QoQ sheet...

Saving to final_complete_analysis.xlsx...

INTEGRATED COMPREHENSIVE ANALYSIS WITH FULL BHK BREAKDOWNS COMPLETE!

COLUMNS CREATED (136 total):
------------------------------------------------------------
1. Base Metrics: 27 columns
   Samples: ['area_name_en', 'Total_Units_Sold', 'Total_Area_Consumed', 'Avg_Rate_per_Sqft', 'Rate_50th_Percentile']...

2. Property Type Breakdowns: 28 columns
   - Units: 4 columns
   - Area: 4 columns
   - Sales: 4 columns
   - Rate stats: 16 columns

3. BHK Breakdowns: 63 columns
   - Units: 9 c

In [1]:
import pandas as pd
df=pd.read_excel('D:\Dubai Data\Location leve summary\Transaction Summary\Transaction_summary.xlsx')
df

<>:2: SyntaxWarning: invalid escape sequence '\D'
<>:2: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Sagar\AppData\Local\Temp\ipykernel_18164\1583045718.py:2: SyntaxWarning: invalid escape sequence '\D'
  df=pd.read_excel('D:\Dubai Data\Location leve summary\Transaction Summary\Transaction_summary.xlsx')


,area_name_en,Total_Units_Sold,Total_Area_Consumed,Avg_Rate_per_Sqft,Rate_50th_Percentile,Rate_75th_Percentile,Rate_90th_Percentile,Total_Sales_Value,Prop_OTHERS_Units_Sold,Prop_OTHERS_Area_Consumed,...,BHK_Area_Sales,PriceRange_Units_Sold,PriceRange_Area_Consumed,PriceRange_Total_Sales,PropType_Price_Units,PropType_Price_Area,PropType_Price_Sales,BHK_Price_Units,BHK_Price_Area,BHK_Price_Sales
0,Al Barsha South Fourth,62855,5.551471e+07,1258.729093,1241.688421,1496.743745,1743.209617,6.474722e+10,2037,6.455848e+06,...,"{'2 BHK':{'<200':119657.9975,'200-500':4243210...","{'<5M':62637,'5M-10M':122,'10M-15M':18,'15M-20...","{'<5M':51785771.20969699,'5M-10M':652884.46213...","{'<5M':60508229742.3755,'5M-10M':768946681.326...","{'Others':{'<5M':1914,'5M-10M':40,'10M-15M':8,...","{'Others':{'<5M':3076253.085206,'5M-10M':38762...","{'Others':{'<5M':2588180393.1440997,'5M-10M':2...","{'2 BHK':{'<5M':8248,'5M-10M':1},'Commercial':...","{'2 BHK':{'<5M':10784375.198219,'5M-10M':1690....","{'2 BHK':{'<5M':11579316788.7144,'5M-10M':5000..."
1,Business Bay,51074,4.946414e+07,2143.213331,2081.971219,2603.786964,3086.446362,1.095817e+11,3046,3.709933e+06,...,"{'1 BHK':{'<200':5811258.219499999,'200-500':1...","{'<5M':49327,'5M-10M':1053,'10M-15M':279,'15M-...","{'<5M':41294316.390236,'5M-10M':2764074.582402...","{'<5M':81452800100.3928,'5M-10M':7032394341.31...","{'Flat':{'<5M':41963,'5M-10M':736,'10M-15M':20...","{'Flat':{'<5M':35005234.735093,'5M-10M':190710...","{'Flat':{'<5M':70834965989.2192,'5M-10M':48819...","{'1 BHK':{'<5M':19863,'5M-10M':14,'10M-15M':1,...","{'1 BHK':{'<5M':15649603.855801998,'5M-10M':17...","{'1 BHK':{'<5M':32159510322.793503,'5M-10M':76..."
2,Marsa Dubai,41685,5.688945e+07,2680.357463,2339.726307,3565.407520,4688.799599,1.426404e+11,3421,3.804902e+06,...,"{'4 BHK':{'<200':600000.0025000001,'200-500':1...","{'<5M':34504,'5M-10M':5690,'10M-15M':941,'15M-...","{'<5M':38202599.34809499,'5M-10M':11347673.658...","{'<5M':76222565152.18831,'5M-10M':38763106897....","{'Flat':{'<5M':31191,'5M-10M':5298,'10M-15M':8...","{'Flat':{'<5M':36611432.288935,'5M-10M':104286...","{'Flat':{'<5M':70813815791.9042,'5M-10M':36116...","{'4 BHK':{'<5M':430,'5M-10M':413,'10M-15M':230...","{'4 BHK':{'<5M':1250803.391845,'5M-10M':142914...","{'4 BHK':{'<5M':1294468532.6765,'5M-10M':30032..."
3,Wadi Al Safa 5,31071,4.745209e+07,1176.587093,1200.693986,1465.791210,1647.231022,5.084756e+10,916,7.763108e+06,...,"{'3 BHK':{'<200':249999.995,'200-500':1822866....","{'<5M':30611,'5M-10M':413,'10M-15M':33,'15M-20...","{'<5M':42805824.99216801,'5M-10M':3754097.5244...","{'<5M':47216425694.51309,'5M-10M':2755012051.1...","{'Flat':{'<5M':29991,'5M-10M':44},'Others':{'<...","{'Flat':{'<5M':39365128.534235,'5M-10M':168506...","{'Flat':{'<5M':45189965101.4141,'5M-10M':25312...","{'3 BHK':{'<5M':7748},'5 BHK':{'<5M':92,'5M-10...","{'3 BHK':{'<5M':13022308.734036},'5 BHK':{'<5M...","{'3 BHK':{'<5M':16975242268.432701},'5 BHK':{'..."
4,Al Merkadh,28933,2.753085e+07,1856.996265,1818.749710,2003.696616,2308.831372,5.066351e+10,496,5.547278e+06,...,"{'1 BHK':{'<200':99999.98190000001,'200-500':1...","{'<5M':28141,'5M-10M':339,'10M-15M':171,'15M-2...","{'<5M':21133681.739001997,'5M-10M':1173849.149...","{'<5M':38670406734.5241,'5M-10M':2242729215.26...","{'Shop':{'<5M':864,'5M-10M':54,'10M-15M':8,'15...","{'Shop':{'<5M':505961.317421,'5M-10M':90632.68...","{'Shop':{'<5M':1622229161.6955,'5M-10M':356673...","{'1 BHK':{'<5M':12051},'2 BHK':{'<5M':5852},'4...","{'1 BHK':{'<5M':8302779.150463},'2 BHK':{'<5M'...","{'1 BHK':{'<5M':14917396869.2966},'2 BHK':{'<5..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,Al Qusais Industrial Third,1,8.181748e+03,700.000000,700.000000,700.000000,700.000000,5.727224e+06,1,8.181748e+03,...,NaN,{'5M-10M':1},{'5M-10M':8181.748028999999},{'5M-10M':5727223.6203},{'Others':{'5M-10M':1}},{'Others':{'5M-10M':8181.748028999999}},{'Others':{'5M-10M':5727223.6203}},NaN,NaN,NaN
173,Dubai Internatio

In [2]:
df['Avg_Rate_per_Sqft'].describe()

count       177.000000
mean       2749.344594
std       18231.920118
min           0.092903
25%         454.464889
50%        1042.411761
75%        1839.374936
max      243316.996583
Name: Avg_Rate_per_Sqft, dtype: float64